# Lab 02 Extra - Banco de Dados Northwind
**Disciplina:** Extração e Preparação de Dados | **Professor:** Luis Aramis

Este é um notebook extra para praticar SQL com um banco de dados clássico: o **Northwind**.
Ele simula uma importadora/exportadora de alimentos gourmet.

## 1. Setup e Download
Vamos baixar o `northwind.db` e conectar o SQLAlchemy.

In [15]:
import pandas as pd
from sqlalchemy import create_engine
import os
import urllib.request

In [20]:
path = '/home/julia/IBMEC/ETL/data-extraction-course/Atividades/data/lab2/northwind.db'

if not os.path.exists(path):
    url = 'https://github.com/jpwhite3/northwind-SQLite3/raw/main/dist/northwind.db'
    urllib.request.urlretrieve(url, path)
    print("Banco baixado!")

engine = create_engine(f"sqlite:///{path}")
print("Conexão estabelecida!")

Conexão estabelecida!


## 2. Mapa do Banco
Quais tabelas temos aqui?

In [ ]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""
df = pd.read_sql(query, engine)
df

,name
0,Categories
1,sqlite_sequence
2,CustomerCustomerDemo
3,CustomerDemographics
4,Customers
5,Employees
6,EmployeeTerritories
7,Order Details
8,Orders
9,Products


## 3. Consultas Básicas
1. Liste os 5 produtos mais caros (`Products`).
2. Liste todos os clientes (`Customers`) que moram no 'Brazil'.

In [23]:
# Produtos mais caros
produtos = 'select * from Products order by UnitPrice desc limit 5'

df_produtos = pd.read_sql(produtos,engine)
df_produtos

,ProductID,ProductName,SupplierID,CategoryID,QuantityPerUnit,UnitPrice,UnitsInStock,UnitsOnOrder,ReorderLevel,Discontinued
0,38,Côte de Blaye,18,1,12 - 75 cl bottles,263.50,17,0,15,0
1,29,Thüringer Rostbratwurst,12,6,50 bags x 30 sausgs.,123.79,0,0,0,1
2,9,Mishi Kobe Niku,4,6,18 - 500 g pkgs.,97.00,29,0,0,1
3,20,Sir Rodney's Marmalade,8,3,30 gift boxes,81.00,40,0,0,0
4,18,Carnarvon Tigers,7,8,16 kg pkg.,62.50,42,0,0,0


In [25]:
# Clientes do Brasil
clientes = "select * from Customers where Country = 'Brazil'"

df_clientes = pd.read_sql(clientes,engine)
df_clientes

,CustomerID,CompanyName,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax
0,COMMI,Comércio Mineiro,Pedro Afonso,Sales Associate,"Av. dos Lusíadas, 23",Sao Paulo,South America,05432-043,Brazil,(11) 555-7647,NaN
1,FAMIA,Familia Arquibaldo,Aria Cruz,Marketing Assistant,"Rua Orós, 92",Sao Paulo,South America,05442-030,Brazil,(11) 555-9857,NaN
2,GOURL,Gourmet Lanchonetes,André Fonseca,Sales Associate,"Av. Brasil, 442",Campinas,South America,04876-786,Brazil,(11) 555-9482,NaN
3,HANAR,Hanari Carnes,Mario Pontes,Accounting Manager,"Rua do Paço, 67",Rio de Janeiro,South America,05454-876,Brazil,(21) 555-0091,(21) 555-8765
4,QUEDE,Que Delícia,Bernardo Batista,Accounting Manager,"Rua da Panificadora, 12",Rio de Janeiro,South America,02389-673,Brazil,(21) 555-4252,(21) 555-4545
5,QUEEN,Queen Cozinha,Lúcia Carvalho,Marketing Assistant,"Alameda dos Canàrios, 891",Sao Paulo,South America,05487-020,Brazil,(11) 555-1189,NaN
6,RICAR,Ricardo Adocicados,Janete Limeira,Assistant Sales Agent,"Av. Copacabana, 267",Rio de Janeiro,South America,02389-890,Brazil,(21) 555-3412,NaN
7,TRADH,Tradição Hipermercados,Anabela Domingues,Sales Representative,"Av. Inês de Castro, 414",Sao Paulo,South America,05634-030,Brazil,(11) 555-2167,(11) 555-2168
8,WELLI,Wellington Importadora,Paula Parente,Sales Manager,"Rua do Mercado, 12",Resende,South America,08737-363,Brazil,(14) 555-8122,NaN


In [27]:
# Clientes do Brasil
clientes = "select * from Customers"

df_clientes = pd.read_sql(clientes,engine)
df_clientes

,CustomerID,CompanyName,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,Western Europe,12209,Germany,030-0074321,030-0076545
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,Central America,05021,Mexico,(5) 555-4729,(5) 555-3745
2,ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,Mataderos 2312,México D.F.,Central America,05023,Mexico,(5) 555-3932,NaN
3,AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,British Isles,WA1 1DP,UK,(171) 555-7788,(171) 555-6750
4,BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,Northern Europe,S-958 22,Sweden,0921-12 34 65,0921-12 34 67
...,...,...,...,...,...,...,...,...,...,...,...
88,WARTH,Wartian Herkku,Pirkko Koskitalo,Accounting Manager,Torikatu 38,Oulu,Scandinavia,90110,Finland,981-443655,981-443655
89,WELLI,Wellington Importadora,Paula Parente,Sales Manager,"Rua do Mercado, 12",Resende,South America,08737-363,Brazil,(14) 555-8122,NaN
90,WHITC,White Clover Markets,Karl Jablonski,Owner,305 - 14th Ave. S. Suite 3B,Seattle,North America,98128,USA,(206) 555-4112,(206) 555-4115
91,WILMK,Wilman Kala,Matti Karttunen,Owner/Marketing Assistant,Keskuskatu 45,Helsinki,Scandinavia,21240,Finland,90-224 8858,90-224 8858


## 4. JOIN: Pedidos e Clientes
Vamos ver quem fez quais pedidos.
Tabelas: `Orders` e `Customers`.
Chave de ligação: `CustomerID`.

In [28]:
pedidos_clientes = '''select o.OrderID, o.CustomerID, o.OrderDate, c.CompanyName, c.ContactName, c.City, c.Region, c.Country
from Orders o
left join Customers c
on o.CustomerID = c.CustomerID'''

df_pedidos_clientes = pd.read_sql(pedidos_clientes,engine)
df_pedidos_clientes


,OrderID,CustomerID,OrderDate,CompanyName,ContactName,City,Region,Country
0,10248,VINET,2016-07-04,Vins et alcools Chevalier,Paul Henriot,Reims,Western Europe,France
1,10249,TOMSP,2016-07-05,Toms Spezialitäten,Karin Josephs,Münster,Western Europe,Germany
2,10250,HANAR,2016-07-08,Hanari Carnes,Mario Pontes,Rio de Janeiro,South America,Brazil
3,10251,VICTE,2016-07-08,Victuailles en stock,Mary Saveley,Lyon,Western Europe,France
4,10252,SUPRD,2016-07-09,Suprêmes délices,Pascale Cartrain,Charleroi,Western Europe,Belgium
...,...,...,...,...,...,...,...,...
16277,26525,WHITC,2012-12-26 04:58:22,White Clover Markets,Karl Jablonski,Seattle,North America,USA
16278,26526,WOLZA,2022-08-05 06:33:39,Wolski Zajazd,Zbyszek Piestrzeniewicz,Warszawa,Eastern Europe,Poland
16279,26527,GOURL,2022-02-09 08:20:12,Gourmet Lanchonetes,André Fonseca,Campinas,South America,Brazil
16280,26528,BLONP,2020-04-27 23:05:30,Blondesddsl père et fils,Frédérique Citeaux,Strasbourg,Western Europe,France


## 5. JOIN Triplo: Detalhes do Pedido
O que tem dentro do pedido 10248?
Caminho: `OrderDetails` -> `Products`.

In [38]:
pedidos_clientes = "select od.OrderID, od.UnitPrice, od.Quantity, od.Discount, Products.ProductName,Products.QuantityPerUnit,Products.UnitsInStock, Products.UnitsOnOrder from 'Order Details' od left join Products on Products.ProductID = od.ProductID where OrderID = '10248'"

df_pedidos_clientes = pd.read_sql(pedidos_clientes,engine)
df_pedidos_clientes

,OrderID,UnitPrice,Quantity,Discount,ProductName,QuantityPerUnit,UnitsInStock,UnitsOnOrder
0,10248,14.0,12,0.0,Queso Cabrales,1 kg pkg.,22,30
1,10248,9.8,10,0.0,Singaporean Hokkien Fried Mee,32 - 1 kg pkgs.,26,0
2,10248,34.8,5,0.0,Mozzarella di Giovanni,24 - 200 g pkgs.,14,0


## 6. Desafio: Total de Vendas por Categoria
Descubra qual Categoria de produtos (`Categories`) gerou mais receita.
Dica: Você vai precisar ligar `Categories` -> `Products` -> `Order Details`.

In [ ]:
#categoria que gerou mais receita 

categoria_produto = "select * from 'Order Details'"

df_pedidos_clientes = pd.read_sql(categoria_produto,engine)
df_pedidos_clientes


,OrderID,ProductID,UnitPrice,Quantity,Discount
0,10248,11,14.00,12,0.0
1,10248,42,9.80,10,0.0
2,10248,72,34.80,5,0.0
3,10249,14,18.60,9,0.0
4,10249,51,42.40,40,0.0
...,...,...,...,...,...
609278,26529,10,31.00,26,0.0
609279,26529,46,12.00,18,0.0
609280,26529,26,31.23,3,0.0
609281,26529,27,43.90,24,0.0


In [ ]:
prod_cat = '''select 
c.CategoryID, c.CategoryName, c.Description, p.ProductID, p.ProductName, p.UnitPrice from Categories c 
left join Products p 
on  p.CategoryID = c.CategoryID'''



query = '''select 
c.CategoryName, c.Description, sum(p.UnitPrice) as valor_categoria from Categories c 
left join Products p 
on  p.CategoryID = c.CategoryID
group by c.CategoryName, c.Description
order by valor_categoria desc '''

df_produtos_categorias  = pd.read_sql(query,engine)
df_produtos_categorias 


#O Produto que gerou mais receita foi: Beverages> Soft drinks, coffees, teas, beers, and ales > 455.75

,CategoryName,Description,valor_categoria
0,Beverages,"Soft drinks, coffees, teas, beers, and ales",455.75
1,Confections,"Desserts, candies, and sweet breads",327.08
2,Meat/Poultry,Prepared meats,324.04
3,Dairy Products,Cheeses,287.30
4,Condiments,"Sweet and savory sauces, relishes, spreads, an...",276.75
5,Seafood,Seaweed and fish,248.19
6,Produce,Dried fruit and bean curd,161.85
7,Grains/Cereals,"Breads, crackers, pasta, and cereal",141.75
